In [ ]:
import pandas as pd
from pathlib import Path
from reed import utils
import tarfile

In [ ]:
config = utils.Config()

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"

## Define formulas

In [ ]:
wwr_pmg11007 = monitoring_data_path/'WWR - PMG-11007'

In [ ]:
files_with_energy_data = {}

for tar_path in wwr_pmg11007.glob("*.tar.gz"):
    try:
        with tarfile.open(tar_path) as tar:
            member = tar.getmembers()[0]  # assume one CSV inside
            # read only first few rows to inspect columns
            df_preview = pd.read_csv(tar.extractfile(member), nrows=5)
            files_with_energy_data[tar_path.name] = "Power_Active" in df_preview.columns
    except Exception as e:
        files_with_energy_data[tar_path.name] = f"Error: {e}"

files_with_energy_data

In [ ]:
def load_wwr(file_number):
    paths = list(wwr_pmg11007.glob("*.gz"))
    wwr_path = paths[file_number]

    wwr_df = pd.read_csv(wwr_path)
    wwr_df["timestamp"] = pd.to_datetime(wwr_df["timestamp"])

    return wwr_df

In [ ]:
def analyse_wwr(wwr_df, keep_columns):
    df = wwr_df
    df = df[keep_columns].copy()
    df["energy_consumption_kWh"] = df["Power_Active"] / 3600
    df_bytool = (
        df.groupby("Tool_Number", as_index=False)
            .agg({
                "energy_consumption_kWh": "sum",
                "Power_Active": "mean"
            })
        )

    return df_bytool

In [ ]:
keep_columns = ["Tool_Number", "Power_Active"]

## First Operation: OF10001

In [ ]:
# operation OF10001 - WWR
file001 = load_wwr(0)
file002 = load_wwr(1) # has energy data
file003 = load_wwr(2)
file004 = load_wwr(3) # has tool data

In [ ]:
file004.info()

In [ ]:
# sort the relevant files by timestamp
file002 = file002.sort_values("timestamp")
file004 = file004.sort_values("timestamp")

In [ ]:
# merge df for file002 and file004
WWR_OF10001_df = pd.merge_asof(file002, file004, on="timestamp", direction="nearest", tolerance=pd.Timedelta("300ms"))
WWR_OF10001_df.head(20)

In [95]:
WWR_OF10001_df.sort_values("timestamp")

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
0,192141848.0,2658105.0,5950.0293,6082.1510,-0.978277,-1260.8372,2025-02-11 08:34:37.609000+00:00,5.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
1,192141851.0,2658105.0,7726.6523,7842.5130,-0.985227,-1343.0791,2025-02-11 08:34:38.609000+00:00,31.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
2,192141853.0,2658105.0,8022.6973,8861.2400,-0.905370,-3762.6992,2025-02-11 08:34:39.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
3,192141855.0,2658105.0,8137.1570,8255.3630,-0.985681,-1392.0126,2025-02-11 08:34:40.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
4,192141857.0,2658105.0,6060.3096,6207.0810,-0.976354,-1341.8284,2025-02-11 08:34:41.609000+00:00,28.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284994,192951307.0,2666017.0,9583.7580,10441.3850,0.917863,4144.1646,2025-02-14 15:47:19.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284995,192951310.0,2666017.0,9648.0020,10485.8570,0.920097,4107.2207,2025-02-14 15:47:20.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284996,192951311.0,2666017.0,6039.5760,6124.7900,-0.986087,-1018.1235,2025-02-14 15:47:21.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
284997,192951313.0,2666017.0,5924.6484,6029.8050,-0.982560,-1121.2013,2025-02-14 15:47:22.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN


In [98]:
192951315.0-192141848.0

809467.0

In [96]:
# results for operation OF10002
WWR_OF10001_df_bytool = analyse_wwr(WWR_OF10001_df, keep_columns)
WWR_OF10001_df_bytool

,Tool_Number,energy_consumption_kWh,Power_Active
0,0.0,30112.564192,2907.553672
1,81.0,3450.542892,12688.411042
2,82.0,4866.516312,10585.775663
3,100.0,16251.491743,7556.880686
4,102.0,16846.361570,9885.395542
5,2424.0,575.641532,10519.337647
6,2427.0,12325.415834,21890.230391
7,2429.0,306.580052,13625.780105
8,2432.0,2699.713919,9435.893309
9,2433.0,9032.701087,11803.166575


In [97]:
WWR_OF10001_df_bytool["energy_consumption_kWh"].sum()

786442.5428941066

## Sceond Operation: OF10002

In [ ]:
# operation OF10002 - WWR
file005 = load_wwr(4)
file006 = load_wwr(5) # has energy data
file007 = load_wwr(6)
file008 = load_wwr(7) # has tool data

In [ ]:
file006.info()

In [ ]:
# sort the relevant files by timestamp
file006 = file006.sort_values("timestamp")
file008 = file008.sort_values("timestamp")

In [ ]:
# merge df for file002 and file004
WWR_OF10002_df = pd.merge_asof(file006, file008, on="timestamp", direction="nearest", tolerance=pd.Timedelta("300ms"))
WWR_OF10002_df.head(20)

In [99]:
WWR_OF10002_df.sort_values("timestamp")

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
0,192951323.0,2666018.0,5868.2910,5983.2940,-0.981857,-1167.4624,2025-02-14 15:47:28.705000+00:00,107.0,0.0,150.0,...,0.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2455.0,NaN
1,192951324.0,2666018.0,5980.8440,6071.0864,-0.985136,-1042.8768,2025-02-14 15:47:29.705000+00:00,107.0,0.0,150.0,...,0.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2455.0,NaN
2,192951326.0,2666018.0,5865.5254,5961.3420,-0.983927,-1064.5231,2025-02-14 15:47:30.705000+00:00,107.0,0.0,150.0,...,0.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2455.0,NaN
3,192951328.0,2666018.0,5895.9316,6022.8257,-0.978931,-1229.8071,2025-02-14 15:47:31.705000+00:00,107.0,0.0,150.0,...,0.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2455.0,NaN
4,192951329.0,2666018.0,5881.6064,6002.3584,-0.979883,-1197.9204,2025-02-14 15:47:32.705000+00:00,107.0,0.0,150.0,...,0.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2455.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1424,192955187.0,2666041.0,6001.6885,6097.2330,-0.983805,-1075.1683,2025-02-14 16:11:12.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1425,192955188.0,2666041.0,6093.3623,6181.7060,-0.984877,-1041.3591,2025-02-14 16:11:13.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1426,192955190.0,2666041.0,6067.3965,6137.4550,-0.988038,-924.6929,2025-02-14 16:11:14.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1427,192955192.0,2666041.0,6056.2505,6141.3926,-0.985866,-1019.0840,2025-02-14 16:11:15.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
192955194.0-192951323.0	

3871.0

In [ ]:
# results for operation OF10002
WWR_OF10002_df_bytool = analyse_wwr(WWR_OF10002_df, keep_columns)
WWR_OF10002_df_bytool

In [ ]:
WWR_OF10002_df_bytool["energy_consumption_kWh"].sum()

In [ ]:
# tools change over time
import matplotlib.pyplot as plt

def plot_energy_vs_time(*dfs, label_prefix="WWR_OF10002_df"):
    plt.figure(figsize=(14,6))

    for i, df in enumerate(dfs, start=1):
        # calculate elapsed time since start of dataset
        df = df.copy()
        df["energy_consumption"] = df["Power_Active"] / 3600
        df["elapsed_seconds"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        
        # plot meaningfully, fx energy_consumption (or Tool_Number)
        plt.plot(df["elapsed_seconds"], df["energy_consumption"], label=f"{label_prefix}{i}")

    plt.title("Energy consumption over elapsed time")
    plt.xlabel("Time since start (seconds)")
    plt.ylabel("Energy consumption (kWh per second)")
    plt.legend()
    plt.show()


# --- USE IT ---
plot_energy_vs_time(WWR_OF10002_df)

